<a href="https://colab.research.google.com/github/angioitoan2409/flood_forecasting/blob/main/radolan_aligned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

BASE_DIR = "/content/drive/MyDrive/flood_forecasting"

RAW_DIR   = f"{BASE_DIR}/raw_data/radolan_events"   # <- moved here (raw RADOLAN pkl files)
MODEL_DIR = f"{BASE_DIR}/model_inputs"              # <- dtm.tif, buildings.tif etc. live here
OUT_DIR   = f"{MODEL_DIR}/radolan_aligned"           # <- reprojected output goes here

os.makedirs(OUT_DIR, exist_ok=True)

print("Raw RADOLAN pkl files found:")
for f in sorted(os.listdir(RAW_DIR)):
    print(" ", f)

Raw RADOLAN pkl files found:
  2017-07-09_radolan.pkl
  2017-07-10_radolan.pkl
  2017-07-11_radolan.pkl
  2019-06-12_radolan.pkl
  2020-08-30_radolan.pkl
  2021-06-29_radolan.pkl
  2021-07-13_radolan.pkl
  2021-07-16_radolan.pkl


In [3]:
import pickle, glob
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin

RADOLAN_CRS = ("+proj=stere +lat_0=90 +lat_ts=60 +lon_0=10 "
               "+a=6370040 +b=6370040 +x_0=0 +y_0=0 +units=m +no_defs")

with rasterio.open(f"{MODEL_DIR}/dtm.tif") as ref:
    dst_transform = ref.transform
    dst_crs = ref.crs
    dst_shape = ref.shape
    domain_mask = ref.read(1) != ref.nodata

def reproject_event(pkl_path, out_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)

    n_t = len(data["timestamps"])
    stack = np.zeros((n_t, *dst_shape), dtype=np.float32)

    for i, (arr, sub_xll, sub_yll, cellsize, nodata) in enumerate(data["compact_data"]):
        n_rows = arr.shape[0]
        # sub_xll/sub_yll are BOTTOM-LEFT corner -> from_origin needs TOP-LEFT (north)
        src_transform = from_origin(sub_xll, sub_yll + n_rows * cellsize, cellsize, cellsize)

        reproject(
            source=arr,
            destination=stack[i],
            src_transform=src_transform,
            src_crs=RADOLAN_CRS,
            src_nodata=nodata,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=0.0,
            resampling=Resampling.bilinear,
        )

    stack[:, ~domain_mask] = 0.0  # zero rain outside domain / under buildings

    np.savez_compressed(
        out_path,
        timestamps=np.array(data["timestamps"], dtype=object),
        event_name=data["event_name"],
        rain_mm=stack,
    )
    return stack

print("Functions loaded.")

Functions loaded.


In [4]:
pkl_files = sorted(glob.glob(f"{RAW_DIR}/*.pkl"))
print(f"Found {len(pkl_files)} pkl files to process.\n")

for pkl_path in pkl_files:
    event_name = os.path.basename(pkl_path).replace("_radolan.pkl", "")
    out_path = f"{OUT_DIR}/{event_name}_aligned.npz"

    if os.path.exists(out_path):
        print(f"Skipping {event_name} — already processed")
        continue

    print(f"Processing {event_name}...")
    stack = reproject_event(pkl_path, out_path)
    print(f"  shape={stack.shape}  max={stack.max():.2f}mm  saved -> {out_path}\n")

print("All events reprojected.")


Found 8 pkl files to process.

Processing 2017-07-09...
  shape=(41, 1600, 1600)  max=7.49mm  saved -> /content/drive/MyDrive/flood_forecasting/model_inputs/radolan_aligned/2017-07-09_aligned.npz

Processing 2017-07-10...
  shape=(41, 1600, 1600)  max=4.80mm  saved -> /content/drive/MyDrive/flood_forecasting/model_inputs/radolan_aligned/2017-07-10_aligned.npz

Processing 2017-07-11...
  shape=(47, 1600, 1600)  max=4.79mm  saved -> /content/drive/MyDrive/flood_forecasting/model_inputs/radolan_aligned/2017-07-11_aligned.npz

Processing 2019-06-12...
  shape=(81, 1600, 1600)  max=7.30mm  saved -> /content/drive/MyDrive/flood_forecasting/model_inputs/radolan_aligned/2019-06-12_aligned.npz

Processing 2020-08-30...
  shape=(350, 1600, 1600)  max=2.70mm  saved -> /content/drive/MyDrive/flood_forecasting/model_inputs/radolan_aligned/2020-08-30_aligned.npz

Processing 2021-06-29...
  shape=(134, 1600, 1600)  max=8.99mm  saved -> /content/drive/MyDrive/flood_forecasting/model_inputs/radolan_ali